In [1]:
%env CUDA_VISIBLE_DEVICES=6

env: CUDA_VISIBLE_DEVICES=6


In [8]:
import json
import csv
import math
import random
from pathlib import Path

import numpy as np
import imageio.v2 as imageio
import genesis as gs
import trimesh
from genesis_demo_sophy_appearance0324_utils import ensure_dir,build_asset_bank
DATASET_ROOT = Path("/data/gaoya/AAA_test_video/Dataset_test/genesis_sim_sophy_0322444")
ASSET_CACHE_DIR = Path("/data/gaoya/AAA_test_video/Dataset_test/genesis_sim_sophy/")
ASSET_CACHE_DIR = ASSET_CACHE_DIR / "_asset_cache"
ASSET_MANIFEST_PATH = ASSET_CACHE_DIR / "asset_manifest.json"

ensure_dir(DATASET_ROOT)
ensure_dir(DATASET_ROOT / "train")
ensure_dir(DATASET_ROOT / "failed_configs")
ensure_dir(ASSET_CACHE_DIR)

In [10]:
asset_bank = build_asset_bank()
backend_used = "gpu"
try:
    gs.init(backend=gs.gpu)
    backend_used = "gpu"
except Exception:
    gs.init(backend=gs.cpu)
    backend_used = "cpu"


[INFO] scanning root=/data/gaoya/dataset/SOPHY_data/bag | found candidate dirs=109
[INFO] scanning root=/data/gaoya/dataset/SOPHY_data/teddy_bear | found candidate dirs=54
[INFO] total raw assets found: 163
[INFO] usable assets: 163
[Genesis] [06:20:53] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [06:20:53] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [06:20:53] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [06:20:53] [INFO] Running on [NVIDIA GeForce RTX 4090] with backend gs.cuda. Device memory: 47.38 GB.
[Genesis] [06:20:53] [INFO] 🚀 Genesis initialized. 🔖 version: 0.4.0, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO


In [11]:
IMG_W, IMG_H = 640, 480
N_SCENES = 5

MAX_OBJECT_PC = 2048
OBJECT_PC_STRIDE = 5
CAMERA_PC_STRIDE = 2

ENABLE_CLOTH = False
CLOTH_MESH_PATH = None
STOP_ON_ERROR = False

SCENE_FAMILY_WEIGHTS = {
    "rigid_mix": 0.4,
    "mpm_mix": 0.3,
    "sph_liquid": 0.3,
}

# =========================
# 数据集 asset 配置
# =========================
SOURCE_DATASET_ROOTS = [
    Path("/data/gaoya/dataset/SOPHY_data/bag"),
    Path("/data/gaoya/dataset/SOPHY_data/teddy_bear"),
]

# USE_DATASET_MESH_OBJECTS = True
# DATASET_OBJECT_PROB = 0.8          # rigid_mix 场景里，一个物体采样为 dataset mesh 的概率
MAX_ASSETS_PER_ROOT = 5          # 调试时可设成小整数
ASSET_CACHE_DIR = Path("/data/gaoya/AAA_test_video/Dataset_test/genesis_sim_sophy/")
ASSET_CACHE_DIR = ASSET_CACHE_DIR / "_asset_cache"
ASSET_MANIFEST_PATH = ASSET_CACHE_DIR / "asset_manifest.json"


TARGET_MESH_SIZE_RANGE = (0.2, 0.5)   # 最长边目标尺寸（米）
SIMPLIFY_MESH_FACE_COUNT = 3000         # None 表示不减面；建议 2000~5000
MIN_VALID_MESH_EXTENT = 1e-5

# 容器：开口朝 -y，相机放在前方（负 y）看进去
# 调整目标：
# 1) 容器尽量放大，提升“物体落在容器内部”的概率
# 2) 改成真正三面体：地面 + 左右墙 + 后墙，前方完全开口
# 3) 配合更保守的出生区和初速度，减少物体从前方飞出
# 增大容器尺寸，提升“物体落在容器内部”的概率
CONTAINER = {

    "half_x": 1.5,# 宽
    "half_y": 1.5,# 深
    "wall_thickness": 0.04,
    "wall_height": 2,# 高
    "front_lip_height": 0.00,
    "floor_thickness": 0.05,
    "center": [0.0, 0.0, 0.0],
}

# 出生区域安全边距：
# 前开口方向（-y）预留更大 buffer，尽量把物体出生点压到容器中后部
SPAWN_FRONT_KEEP_OUT = 0.42
SPAWN_BACK_KEEP_OUT = 0.10
SPAWN_SIDE_KEEP_OUT = 0.06

# =========================
# 数据集 mesh 的朝向修正
# =========================
# 说明：
# teddy_bear 很可能是 y-up 的模型，在 z-up 世界里会平躺。
# 这里先用绕 x 轴 -90° 进行扶正。
# 如果你发现扶正后仍然不对，把 teddy_bear 改成 [0.0, -math.pi / 2.0, 0.0] 试一下。
DATASET_BASE_EULER_BY_DATASET = {
    "teddy_bear":  [0.0, -math.pi / 2.0, 0.0] ,
    "bag": [0.0, 0.0, 0.0],
}

# =========================
# rigid 运动模式
# 目标：优先保证物体尽量落在容器内部，因此：
# - 上方下落 / 上方轻抛占绝大多数
# - 左右侧抛显著减少且速度收敛
# =========================
RIGID_MOTION_WEIGHTS = {
    "top_drop": 0.70,
    "top_toss": 0.24,
    "side_throw_left": 0.03,
    "side_throw_right": 0.03,
}

TOP_DROP_Z_RANGE = (1.05, 1.45)
TOP_TOSS_Z_RANGE = (1.00, 1.38)
SIDE_THROW_Z_RANGE = (0.62, 0.88)

TOP_DROP_VXY = 0.06
TOP_TOSS_VX = 0.30
TOP_TOSS_VY = 0.16
TOP_TOSS_VZ_RANGE = (-0.90, -0.25)    # 向下

SIDE_THROW_VX_RANGE = (0.95, 1.40)    # 朝容器中心，但更保守
SIDE_THROW_VY_RANGE = (0.08, 0.26)    # 统一偏向 +y，把物体往容器后部送
SIDE_THROW_VZ_RANGE = (0.45, 0.90)    # 减少过高抛射

TOP_DROP_ANGVEL = 1.5
TOP_TOSS_ANGVEL = 3.0
SIDE_THROW_ANGVEL = 3.5

USE_DATASET_MESH_OBJECTS = True
DATASET_OBJECT_PROB = 1.0   # 让 rigid_mix 场景尽量都用 SOURCE_DATASET_ROOTS 里的物体
MAX_ASSETS_PER_ROOT = None  # 不再只取前5个，真正把 bag / teddy_bear 都放进来

USE_TEXTURED_DATASET_MESH = True
N_BACKGROUND_PROPS_RANGE = (2, 5)
BACKGROUND_PANEL_Y = 1.10
BACKGROUND_SIDE_X = 1.05
BACKGROUND_Z_RANGE = (0.05, 0.55)



manifest = {
    "dataset_name": "genesis_sim_v3",
    "split": "train",
    "n_scenes_requested": N_SCENES,
    "image_size": [IMG_W, IMG_H],
    "backend_used": backend_used,
    "scene_families": SCENE_FAMILY_WEIGHTS,
    "source_dataset_roots": [str(x) for x in SOURCE_DATASET_ROOTS],
    "use_dataset_mesh_objects": USE_DATASET_MESH_OBJECTS,
    "dataset_object_prob": DATASET_OBJECT_PROB,
    "n_usable_dataset_assets": len(asset_bank),
    "notes": [
        "Uses z-up convention.",
        "Container has back/side walls and a low front lip, so the camera is not occluded.",
        "Rigid scenes can mix dataset mesh objects and procedural primitives.",
        "MPM/SPH scenes remain procedural for stability.",
        "Single-scene failure is recorded instead of aborting the whole run."
    ],
    "scenes": [],
    "failed_scenes": [],
}

In [12]:
from genesis_demo_sophy_appearance0324_utils import ensure_dir,build_asset_bank,export_scene,sample_scene_cfg
try:
    for sid in range(N_SCENES):
        scene_cfg = sample_scene_cfg(sid, asset_bank=asset_bank)

        try:
            print(f"[RUN ] {scene_cfg['scene_id']} | family={scene_cfg['family']}")
            meta = export_scene(scene_cfg)
            manifest["scenes"].append(meta)
            print(
                f"[ OK ] {scene_cfg['scene_id']} | family={scene_cfg['family']} "
                f"| dataset_mesh={meta['num_dataset_mesh_objects']}/{meta['num_objects']}"
            )

        except Exception as e:
            err_info = {
                "scene_id": scene_cfg["scene_id"],
                "family": scene_cfg["family"],
                "seed": scene_cfg["seed"],
                "error": str(e),
            }
            manifest["failed_scenes"].append(err_info)

            with open(DATASET_ROOT / "failed_configs" / f"{scene_cfg['scene_id']}.json", "w", encoding="utf-8") as f:
                json.dump(
                    {
                        "scene_cfg": scene_cfg,
                        "error": str(e),
                    },
                    f,
                    ensure_ascii=False,
                    indent=2,
                )

            print(f"[FAIL] {scene_cfg['scene_id']} | family={scene_cfg['family']} | err={e}")

            if STOP_ON_ERROR:
                raise

finally:
    with open(DATASET_ROOT / "dataset_manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    try:
        gs.destroy()
    except Exception:
        pass

[RUN ] train_scene_000000 | family=rigid_mix
[Genesis] [06:24:10] [INFO] Scene <f3d04d9> created.
[Genesis] [06:24:10] [INFO] Adding <gs.RigidEntity>. idx: 0, uid: <740010b>, morph: <gs.morphs.Box>, material: <gs.materials.Rigid>.
[Genesis] [06:24:10] [INFO] Adding <gs.RigidEntity>. idx: 1, uid: <6a2365b>, morph: <gs.morphs.Box>, material: <gs.materials.Rigid>.
[Genesis] [06:24:10] [INFO] Adding <gs.RigidEntity>. idx: 2, uid: <d51ac9e>, morph: <gs.morphs.Box>, material: <gs.materials.Rigid>.
[Genesis] [06:24:10] [INFO] Adding <gs.RigidEntity>. idx: 3, uid: <d944634>, morph: <gs.morphs.Box>, material: <gs.materials.Rigid>.
[Genesis] [06:24:10] [INFO] Adding <gs.RigidEntity>. idx: 4, uid: <eb0d7ba>, morph: <gs.morphs.Mesh(file='/data/gaoya/dataset/SOPHY_data/teddy_bear/simulation_data/train/teddy_bear/ae_015__0/material.obj')>, material: <gs.materials.Rigid>.
[Genesis] [06:24:12] [INFO] Convex hull is not accurate enough for collision detection (0.614). Falling back to more expensive con

KeyboardInterrupt: 